# SAC Irrigation Training — v2.10 E4 (gamma reduction pilot, Kaggle)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q critic (v2.7 architecture)
**Replay buffer:** SB3 standard 1-step `ReplayBuffer` (NOT the E3 NStepReplayBuffer)
**Single change vs v2.7:** `gamma = 0.99` → `gamma = 0.98`

## Why E4 exists

E2 (TQC + k=5) and E3 (TQC + k=5 + custom n-step buffer) both failed. E3 cascaded faster than E2 because the `NStepReplayBuffer` drops the soft-Bellman entropy bonus on the intermediate steps and uses gamma^1 instead of gamma^n in the bootstrap — both biases that systematically deflate the target.

E4 attacks the same hypothesis (the cascade is driven by bootstrap leverage `1/(1-gamma) ≈ 100` over a 93-day horizon) but at the algorithm level instead of through a custom buffer: lower gamma directly.

- `gamma = 0.99` (v2.7): cascades at step ~156k.
- `gamma = 0.98` (E4): bootstrap leverage halved. **This run.**
- `gamma = 0.97`: fallback if E4 fails.

## Acceptance criterion

`|q_inflation_pct| < 20%` at step 250k AND in-distribution (dry/moderate) yields within ~3% of v2.7's published numbers.

## Early-kill rule

If at any checkpoint past step 150k you see BOTH `q_inflation_pct > 100%` AND `actor/std/spatial < 0.15`, stop the run.

## Kaggle environment

- GPU: T4 x 1 (Accelerator: GPU T4 x2 reduces to T4 x 1 for SB3)
- ~2-2.5 hours per 250k-step run
- WandB API key must be added under **Settings -> Secrets** as `WANDB_API_KEY`


In [ ]:
# Cell 1: Clone repo and install deps.
import subprocess, sys, os

WORK_ROOT = '/kaggle/working'
REPO_DIR  = f'{WORK_ROOT}/thesis'

if os.path.exists(REPO_DIR):
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', REPO_DIR],
    check=True
)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install SB3 — sb3-contrib is NOT required for E4 (no TQC).
subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0',
     'gymnasium', 'wandb', 'pytest'],
    check=True
)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Kaggle Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add WANDB_API_KEY under Settings -> Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Cell 3: Pre-training validation.
#
# NOTE: TQC critic tests are intentionally NOT run here - E4 uses SAC, not TQC.

import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 architecture)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'V2.7 FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1 minute)...')
from src.rl.train_v210_e4 import train_sac_e4
_ = train_sac_e4(
    seed=999,
    output_dir='/kaggle/working/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')


In [ ]:
# Cell 4: Full 250k training (SAC v2.10 E4, gamma=0.98).
# ~2-2.5 hours on Kaggle T4.
#
# Start with SEED=0.  Expand seeds only after seed-0 results meet the
# acceptance criterion.

SEED  = 0       # CHANGE per session
GAMMA = 0.98    # E4 primary.  Set to 0.97 for the fallback experiment.

from src.rl.train_v210_e4 import train_sac_e4

model = train_sac_e4(
    seed=SEED,
    output_dir='/kaggle/working/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=GAMMA,
)
print('Training complete.')


In [ ]:
# Cell 5: Trim the run dir for the Kaggle output (no replay buffer).
import shutil, os

src = f'/kaggle/working/results/rl/sac_v210_e4_seed{SEED}'
rb  = f'{src}/replay_buffer_latest.pkl'
if os.path.exists(rb):
    os.remove(rb)
    print(f'Removed: {rb}')

print('\nFinal output tree:')
for root, _, files in os.walk(src):
    for f in files:
        p = os.path.join(root, f)
        size = os.path.getsize(p)
        rel  = os.path.relpath(p, src)
        print(f'  {rel}  ({size/1024:.1f} KB)')


In [ ]:
# Cell 6: Post-training 9-cell evaluation (SAC eval path).
import subprocess, sys

model_path = f'/kaggle/working/results/rl/sac_v210_e4_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',     'eval',
    '--model',    model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

print('\nEvaluating on 9-cell grid (noisy forecast, seed=42)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',       'eval',
    '--model',      model_path,
    '--scenario',   'all',
    '--budget',     'all',
    '--forecast',   'noisy',
    '--noise-seed', '42',
], capture_output=False)
if r.returncode != 0:
    print('Noisy-forecast eval failed; perfect-forecast only.')


In [ ]:
# Cell 7: Q-inflation trajectory plot.
import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'/kaggle/working/results/rl/sac_v210_e4_seed{SEED}/bias_ratio_log.csv'
df = pd.read_csv(csv_path)
print(df.tail(10))

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax = axes[0]
ax.plot(df['step'], df['q_pred_mean'],  '-o', label='Q_pred_mean',                  color='C0')
ax.plot(df['step'], df['q_structural'], '-s', label='Q_structural (theoretical)',   color='C1')
ax.axhline(0, color='k', linestyle=':', alpha=0.3)
ax.set_ylabel('Q value')
ax.set_title(f'v2.10 E4 (SAC, gamma={GAMMA}) seed {SEED} - Q calibration')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(df['step'], df['q_inflation_pct'], '-o', color='C3', label='q_inflation_pct')
ax.axhline(0.0,   color='k',      linestyle='-',  alpha=0.4, label='ideal')
ax.axhline(20.0,  color='g',      linestyle=':',  alpha=0.6, label='acceptance threshold (20%)')
ax.axhline(-20.0, color='g',      linestyle=':',  alpha=0.6)
ax.axhline(50.0,  color='orange', linestyle=':',  alpha=0.6, label='cascade onset (50%)')
ax.axhline(-50.0, color='orange', linestyle=':',  alpha=0.6)
ax.axhline(200.0, color='r',      linestyle=':',  alpha=0.6, label='full cascade (200%)')
ax.axhline(-200.0,color='r',      linestyle=':',  alpha=0.6)
ax.set_ylabel('Q_inflation %')
ax.set_xlabel('training step')
ax.legend(loc='best', fontsize='small')
ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = f'/kaggle/working/results/rl/sac_v210_e4_seed{SEED}/q_inflation_trajectory.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved plot: {plot_path}')
